In [44]:
import torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
block_size = 8
batch_size = 4
vocab_size = 80
max_iters = 10000 
learning_rate = 3e-4

In [3]:
with open("./data/wizard_of_oz.txt",'r', encoding='utf-8') as f:
    text = f.read()

In [29]:
print(len(text))

232316


In [5]:
chars = sorted(list(set(text)))
print(len(chars))
print(''.join(chars))

81

 !"&'()*,-.0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxyz﻿


In [6]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: [itos[i] for i in l] 

In [7]:
print(encode(text[:10]))
print(''.join(decode(encode(text[:10]))))

[80, 1, 1, 28, 39, 42, 39, 44, 32, 49]
﻿  DOROTHY


In [8]:
data = torch.tensor(encode(text),dtype=torch.long)
print(data[:100])

tensor([80,  1,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,
         1, 47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26,
        49,  0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,
         0,  0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1,
        47, 33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1,
        36, 25, 38, 28,  1, 39, 30,  1, 39, 50])


In [9]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

In [10]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for i in range(block_size):
    context = x[:i+1]
    target = y[i]
    print(f"For context {context} the target is {target}")

For context tensor([80]) the target is 1
For context tensor([80,  1]) the target is 1
For context tensor([80,  1,  1]) the target is 28
For context tensor([80,  1,  1, 28]) the target is 39
For context tensor([80,  1,  1, 28, 39]) the target is 42
For context tensor([80,  1,  1, 28, 39, 42]) the target is 39
For context tensor([80,  1,  1, 28, 39, 42, 39]) the target is 44
For context tensor([80,  1,  1, 28, 39, 42, 39, 44]) the target is 32


##### @ get_batch Function

In [11]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(0,len(data)-block_size,size=(batch_size,)) # block_size=8 , batch_size=4
    print(ix)
    x = torch.stack([data[i:block_size+i] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

In [12]:
x, y = get_batch('train')
print(f"inputs\n{x}")
print(f"targets\n{y}")

tensor([ 98928, 183372,   8931,  19285])
inputs
tensor([[62, 71, 65,  1, 72, 54, 62, 57],
        [67,  1, 65, 68, 75, 58, 57,  1],
        [62, 60, 60, 65, 58, 57,  1, 68],
        [65,  1, 73, 61, 58,  1, 72, 56]])
targets
tensor([[71, 65,  1, 72, 54, 62, 57,  1],
        [ 1, 65, 68, 75, 58, 57,  1, 28],
        [60, 60, 65, 58, 57,  1, 68, 67],
        [ 1, 73, 61, 58,  1, 72, 56, 58]])


#### @ BiGramLanguageModel

In [17]:
class bigramlanguagemodel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)
        
    def forward_pass(self,index,targets=None):
        logits = self.token_embedding_table(index)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits,targets)

        return logits,loss
    
    def generate(self,index,max_new_tokens):  
        for _ in range(max_new_tokens):
            logits, _ = self.forward_pass(index) 
            logits = logits[:,-1,:] # becomes (B,C)
            probs = F.softmax(logits,dim=-1)
            next_index = torch.multinomial(probs,num_samples=1) # (B,1)
            index = torch.cat((index,next_index),dim=1) # (B,T+1) 
        return index


In [37]:
# This code generates garbage tokens(characters) right now as we still haven't trained the model yet.
model = bigramlanguagemodel(vocab_size=vocab_size)
m = model.to(device)
 
context = torch.zeros(size=(1,1),dtype=torch.long,device=device)
generated_chars = decode(m.generate(context,500)[0].tolist())
generated_chars = ''.join(generated_chars)
print(generated_chars)


eA[.Phph?X!lP-eh?7,Dl7]D _AhFIF3BGuL1wWuqRm5VVEN4t(Is!TZdjV"mMWaNFOG4jdJ9ZNhI*.5ZZ45.PrFWTPI--TJ:_&dJNhT29dydy*(6yRW*,[I'y0Tne[C'0!*PYFMkwseHksHBX;DMDaQ22N!F_3IH]THk(6i7QO0'0T)U5ks6Q)im'8&'8JWX4'-8
Rwy_tgVB58!6lN'-Zs*nR5myq2.76"C DfObq!hmM"zsi4mxm.bW';6Rx !!y&JjUsv5KvCF_5bbRmwNXZYctwNXKjOYw
Rxax5Q]ctQ
Rm(EWabE0yRm]FemyAX4
HA_"-g)i.Bs:9wZ;]oZtr*rt-;PJ]FPyK
G(PaN7
AEtl3H)YRoKsHzW' i]0E7* ?q_tHBVJNThcEtgH,jCeaYR;m8&Su3BDb]9L&2]0AEtzhDl
[:FH)6w:,DzyR5_59!KtwBurcwqR;Cjg-8&.51!.Z?zPqRUydjjejmEtgzMNE"D
